# Bank Statement Transaction Extraction with Llama Vision

Specialized notebook for extracting transactional data from bank statements using Llama-3.2-Vision-Instruct.

**Key Features:**
- Handles empty debit/credit cells correctly
- Preserves row-by-row transaction structure
- Calculates running balances
- Validates transaction integrity

# Bank Statement Extraction with Evaluation - Llama Vision

Enhanced notebook for extracting and evaluating bank statement data using Llama-3.2-Vision-Instruct.

## Updates for Evaluation Data Integration

### ✅ New Features:
1. **Multi-Bank Processing**: Process all Big 4 Australian bank statements (CommBank, ANZ, NAB, Westpac)
2. **Ground Truth Integration**: Load and compare with `evaluation_data/ground_truth.csv`
3. **Evaluation Metrics**: 
   - Bank name recognition accuracy
   - Account holder extraction accuracy
   - Transaction count matching
   - Transaction date overlap analysis
4. **Comprehensive Reporting**:
   - Overall performance score
   - Field-level accuracy metrics
   - Per-bank detailed results
   - CSV export of evaluation results

### 📁 Data Sources:
- **Images**: `evaluation_data/` directory containing:
  - `commbank_statement_001.png`
  - `anz_statement_001.png`
  - `nab_statement_001.png`
  - `westpac_statement_001.png`
- **Ground Truth**: `evaluation_data/ground_truth.csv` with expected values

### 🎯 Evaluation Workflow:
1. Load ground truth data for bank statements
2. Process each bank statement image
3. Extract transaction data and account details
4. Compare extracted vs expected values
5. Generate evaluation report with accuracy metrics
6. Export results to CSV for further analysis

In [1]:
# Configuration
MODEL_PATH = "/home/jovyan/nfs_share/models/Llama-3.2-11B-Vision-Instruct"

# Big 4 Australian Bank Statement Images
BANK_STATEMENT_IMAGES = {
    "CommBank": "evaluation_data/commbank_statement_001.png",
    "ANZ": "evaluation_data/anz_statement_001.png", 
    "NAB": "evaluation_data/nab_statement_001.png",
    "Westpac": "evaluation_data/westpac_statement_001.png"
}

# Single statement for quick testing
STATEMENT_IMAGE_PATH = BANK_STATEMENT_IMAGES["CommBank"]

# Ground truth CSV for evaluation
GROUND_TRUTH_CSV = "evaluation_data/ground_truth.csv"

# Model loading settings - NO QUANTIZATION BY DEFAULT (H200 has plenty of memory)
USE_QUANTIZATION = False  # H200 doesn't need quantization
LOAD_IN_8BIT = False  # 8-bit causes inference issues with bitsandbytes  
DEVICE_MAP = "auto"  # Automatic device mapping
MAX_NEW_TOKENS = 3000  # Increased for full statement extraction

In [2]:
import torch
from transformers import MllamaForConditionalGeneration, AutoProcessor
from PIL import Image
import json
import re
from typing import Dict, List, Any, Optional, Tuple
import pandas as pd
from datetime import datetime
from decimal import Decimal, InvalidOperation
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Initialize model and processor
print("Loading Llama Vision model for bank statement processing...")

# Load model WITHOUT quantization by default
# The bitsandbytes 8-bit quantization can cause RuntimeError during inference
if USE_QUANTIZATION and LOAD_IN_8BIT:
    print("⚠️ Loading with 8-bit quantization (may cause inference errors)")
    model = MllamaForConditionalGeneration.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map=DEVICE_MAP,
        load_in_8bit=True,
    )
else:
    print("✅ Loading model without quantization (recommended for H200)")
    model = MllamaForConditionalGeneration.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map=DEVICE_MAP,
    )

processor = AutoProcessor.from_pretrained(MODEL_PATH)

print("✅ Model loaded successfully")
print(f"📊 Device: {next(model.parameters()).device}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name()}")
    print(f"💾 Memory: {torch.cuda.memory_allocated()/1e9:.2f}GB allocated")

Loading Llama Vision model for bank statement processing...
✅ Loading model without quantization (recommended for H200)


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Model loaded successfully
📊 Device: cuda:0
🎮 GPU: NVIDIA H200
💾 Memory: 9.76GB allocated


In [4]:
# Specialized bank statement extraction prompts
BANK_STATEMENT_PROMPTS = {
    "structured_transactions": """Analyze this bank statement image and extract ALL transactions as a structured table.

CRITICAL INSTRUCTIONS:
1. Treat the statement as a TABLE with distinct columns
2. Each transaction is ONE ROW in the table
3. Empty cells MUST be marked as null or empty string
4. PRESERVE the exact row structure - do not skip or combine rows

For EACH transaction row, extract:
- Date (transaction date)
- Description (transaction description/details)
- Debit (amount if it's a debit/withdrawal, otherwise null)
- Credit (amount if it's a credit/deposit, otherwise null)  
- Balance (running balance after this transaction)

Return as JSON:
{
  "account_details": {
    "account_number": "...",
    "statement_period": "...",
    "opening_balance": "...",
    "closing_balance": "..."
  },
  "transactions": [
    {
      "date": "DD/MM/YYYY",
      "description": "...",
      "debit": "100.00" or null,
      "credit": null or "200.00",
      "balance": "1234.56"
    }
  ]
}

IMPORTANT: A transaction will have EITHER a debit OR a credit value, never both. The other MUST be null.""",

    "row_by_row_extraction": """Extract the transaction table from this bank statement image ROW BY ROW.

CRITICAL: This is a TABLE where each row represents one transaction.

For the transaction table:
1. Identify ALL column headers (typically: Date, Description, Debit, Credit, Balance)
2. For EACH row in the table, extract ALL cells
3. If a cell is EMPTY (like debit column for a credit transaction), mark it as "EMPTY"
4. Preserve EXACT row order - do not skip any rows

Format your response EXACTLY as:
COLUMNS: Date | Description | Debit | Credit | Balance
ROW 1: [date] | [description] | [amount or EMPTY] | [amount or EMPTY] | [balance]
ROW 2: [date] | [description] | [amount or EMPTY] | [amount or EMPTY] | [balance]
[continue for all rows]

Remember: Each transaction has EITHER debit OR credit, so one will always be EMPTY.""",

    "csv_with_empty_cells": """Convert the bank statement transaction table to CSV format.

RULES FOR BANK STATEMENTS:
1. First line: Date,Description,Debit,Credit,Balance
2. Each transaction is one line
3. Empty debit/credit cells must be represented as empty (,,)
4. DO NOT skip empty cells - preserve the column structure
5. Amounts should not include currency symbols

Example format:
Date,Description,Debit,Credit,Balance
01/03/2024,Opening Balance,,,5000.00
02/03/2024,ATM Withdrawal,100.00,,4900.00
03/03/2024,Salary Deposit,,3500.00,8400.00

Extract ALL transactions maintaining this exact structure.""",

    "detailed_with_metadata": """Carefully analyze this bank statement and extract all information.

Extract:
1. HEADER INFORMATION:
   - Bank name
   - Account holder name
   - Account number (masked is ok)
   - Statement period (from date to date)
   - Opening balance
   - Closing balance

2. TRANSACTION TABLE (treat as structured table):
   For each row in the transaction table:
   - Date
   - Transaction description/particulars
   - Debit amount (if present, otherwise "nil")
   - Credit amount (if present, otherwise "nil")
   - Running balance

3. SUMMARY (if present):
   - Total debits
   - Total credits
   - Number of transactions

CRITICAL: Preserve the table structure. Each transaction occupies exactly one row.
Mark empty cells explicitly as "nil" or "empty".""",

    "validation_focused": """Extract bank statement transactions with validation checks.

For each transaction in the table:
1. Date (format: DD/MM/YYYY or MM/DD/YYYY)
2. Description (full text)
3. Debit (negative/outgoing amount or NULL)
4. Credit (positive/incoming amount or NULL)
5. Balance (running balance)

VALIDATION RULES:
- Each row must have EITHER debit OR credit, not both
- Balance should equal: previous_balance - debit + credit
- All amounts must be positive numbers (no negative signs)
- Dates must be in chronological order

Return as JSON with a 'validation' field indicating any issues found."""
}

In [5]:
# Enhanced prompts for complex bank statements with multiple sections
COMPLEX_STATEMENT_PROMPTS = {
    "full_statement_extraction": """Analyze this complete bank statement image that contains multiple sections.

DOCUMENT STRUCTURE:
1. HEADER/ACCOUNT SUMMARY (top section)
2. TRANSACTION TABLE (main middle section)  
3. SUMMARY/TOTALS (bottom section)

Extract the following:

SECTION 1 - ACCOUNT INFORMATION (from top of statement):
- Bank name and logo area
- Account holder name
- Account number (may be partially masked)
- Statement period (e.g., "01 Mar 2024 to 31 Mar 2024")
- Opening balance
- Total deposits/credits for period
- Total withdrawals/debits for period
- Closing balance

SECTION 2 - TRANSACTION TABLE:
Focus on the main table with columns. For EACH transaction row:
- Date (transaction date)
- Description/Particulars (transaction details)
- Debit/Withdrawal (amount if debit, else null)
- Credit/Deposit (amount if credit, else null)
- Balance (running balance after transaction)

CRITICAL RULES FOR THE TABLE:
- Each row is ONE transaction
- A transaction has EITHER debit OR credit, never both
- Empty cells must be marked as null
- Preserve exact row order

SECTION 3 - SUMMARY (if present at bottom):
- Total number of transactions
- Sum of all debits
- Sum of all credits
- Final closing balance
- Any notes or messages

Return as structured JSON with all three sections clearly separated.""",

    "table_focused_with_context": """Extract the transaction table from this bank statement, understanding it exists within a larger document.

IMPORTANT: This statement has multiple sections, but focus primarily on the TRANSACTION TABLE in the middle.

Before the table, note:
- Statement date range (if visible)
- Opening balance (if shown above table)

For the TRANSACTION TABLE:
1. Identify exact column headers (Date, Description, Debit, Credit, Balance, etc.)
2. Extract EVERY row maintaining structure
3. Empty cells (like debit for a deposit) = "EMPTY"
4. Do not confuse summary rows with transactions

After the table, note:
- Any summary totals shown
- Closing balance

Format as:
STATEMENT PERIOD: [dates]
OPENING BALANCE: [amount]

TABLE COLUMNS: Date | Description | Debit | Credit | Balance
ROW 1: [values with EMPTY for blank cells]
ROW 2: [values with EMPTY for blank cells]
[all rows]

SUMMARY:
Total Debits: [amount]
Total Credits: [amount]
Closing Balance: [amount]""",

    "hierarchical_extraction": """Process this multi-section bank statement hierarchically.

STEP 1 - Identify document sections:
- Where does the header/account info end?
- Where does the transaction table begin and end?
- Is there a summary section at the bottom?

STEP 2 - Extract header information:
{
  "header": {
    "bank": "...",
    "account_holder": "...",
    "account_number": "...",
    "period": "...",
    "opening_balance": "..."
  }
}

STEP 3 - Extract transaction table:
{
  "transactions": [
    {
      "row_number": 1,
      "date": "...",
      "description": "...",
      "debit": null or amount,
      "credit": null or amount,
      "balance": "..."
    }
  ]
}

STEP 4 - Extract summary if present:
{
  "summary": {
    "total_transactions": ...,
    "total_debits": "...",
    "total_credits": "...",
    "closing_balance": "..."
  }
}

Return complete JSON combining all sections."""
}

In [6]:
def extract_bank_statement(
    image_path: str, 
    prompt_type: str = "structured_transactions",
    custom_prompt: str = None,
    temperature: float = 0.1
) -> Dict[str, Any]:
    """
    Extract bank statement data from an image.
    
    Args:
        image_path: Path to bank statement image
        prompt_type: Type of extraction prompt
        custom_prompt: Optional custom prompt
        temperature: Generation temperature (lower = more deterministic)
    
    Returns:
        Dictionary with extraction results
    """
    # Load image
    image = Image.open(image_path)
    
    # Select prompt
    prompt = custom_prompt if custom_prompt else BANK_STATEMENT_PROMPTS.get(
        prompt_type, BANK_STATEMENT_PROMPTS["structured_transactions"]
    )
    
    # Prepare inputs
    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": prompt}
        ]}
    ]
    
    # Apply chat template
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    
    # Process inputs
    inputs = processor(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(model.device)
    
    # Generate response
    print("🔍 Analyzing bank statement...")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=temperature,
            top_p=0.95
        )
    
    # Decode response
    response = processor.decode(output[0], skip_special_tokens=True)
    
    # Extract assistant response
    if "assistant" in response:
        response = response.split("assistant")[-1].strip()
    
    return {
        "raw_response": response,
        "prompt_type": prompt_type,
        "image_path": image_path,
        "extraction_time": datetime.now().isoformat()
    }

In [7]:
def parse_statement_json(response: str) -> Optional[Dict]:
    """
    Parse JSON response from bank statement extraction.
    """
    try:
        # Find JSON in response
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            data = json.loads(json_match.group())
            return data
    except json.JSONDecodeError as e:
        print(f"❌ JSON parsing error: {e}")
    return None

def parse_row_format(response: str) -> pd.DataFrame:
    """
    Parse row-by-row format response into DataFrame.
    """
    lines = response.strip().split('\n')
    
    # Find column headers
    columns = None
    rows = []
    
    for line in lines:
        if line.startswith('COLUMNS:'):
            columns = [col.strip() for col in line.replace('COLUMNS:', '').split('|')]
        elif line.startswith('ROW'):
            # Extract row data
            row_data = line.split(':', 1)[1] if ':' in line else line
            values = [val.strip() for val in row_data.split('|')]
            
            # Replace EMPTY with None
            values = [None if v.upper() == 'EMPTY' else v for v in values]
            rows.append(values)
    
    if columns and rows:
        return pd.DataFrame(rows, columns=columns)
    return None

def parse_csv_format(response: str) -> pd.DataFrame:
    """
    Parse CSV format with proper handling of empty cells.
    """
    try:
        lines = response.strip().split('\n')
        csv_lines = []
        
        for line in lines:
            # Only include lines that look like CSV
            if ',' in line and not line.startswith('#'):
                csv_lines.append(line)
        
        if csv_lines:
            from io import StringIO
            csv_text = '\n'.join(csv_lines)
            df = pd.read_csv(StringIO(csv_text))
            
            # Replace NaN with None for clarity
            df = df.where(pd.notnull(df), None)
            return df
    except Exception as e:
        print(f"❌ CSV parsing error: {e}")
    return None

In [8]:
def validate_transactions(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Validate bank statement transactions.
    
    Returns:
        Dictionary with validation results and issues
    """
    issues = []
    
    # Check for required columns
    required_cols = ['date', 'description', 'debit', 'credit', 'balance']
    df_cols_lower = [col.lower() for col in df.columns]
    
    for req_col in required_cols:
        if req_col not in df_cols_lower:
            issues.append(f"Missing required column: {req_col}")
    
    # Normalize column names
    df.columns = [col.lower() for col in df.columns]
    
    # Validation checks
    for idx, row in df.iterrows():
        # Check debit/credit exclusivity
        has_debit = row.get('debit') not in [None, '', 'nil', 'empty', 'EMPTY']
        has_credit = row.get('credit') not in [None, '', 'nil', 'empty', 'EMPTY']
        
        if has_debit and has_credit:
            issues.append(f"Row {idx}: Has both debit and credit values")
        elif not has_debit and not has_credit:
            # Check if it's not an opening/closing balance row
            desc = str(row.get('description', '')).lower()
            if 'opening' not in desc and 'closing' not in desc:
                issues.append(f"Row {idx}: Missing both debit and credit values")
    
    # Calculate totals
    total_debits = 0
    total_credits = 0
    
    for _, row in df.iterrows():
        try:
            debit_val = row.get('debit')
            if debit_val and str(debit_val).replace('.', '').replace(',', '').isdigit():
                total_debits += float(str(debit_val).replace(',', ''))
            
            credit_val = row.get('credit')
            if credit_val and str(credit_val).replace('.', '').replace(',', '').isdigit():
                total_credits += float(str(credit_val).replace(',', ''))
        except:
            pass
    
    return {
        "valid": len(issues) == 0,
        "issues": issues,
        "total_debits": total_debits,
        "total_credits": total_credits,
        "net_change": total_credits - total_debits,
        "transaction_count": len(df)
    }

In [9]:
def calculate_running_balance(df: pd.DataFrame, opening_balance: float = None) -> pd.DataFrame:
    """
    Calculate or verify running balance for transactions.
    """
    df = df.copy()
    
    # Normalize column names
    df.columns = [col.lower() for col in df.columns]
    
    # Convert amounts to float
    def to_float(val):
        if val in [None, '', 'nil', 'empty', 'EMPTY', 'null']:
            return 0.0
        try:
            return float(str(val).replace(',', '').replace('$', ''))
        except:
            return 0.0
    
    df['debit_amount'] = df['debit'].apply(to_float)
    df['credit_amount'] = df['credit'].apply(to_float)
    df['balance_amount'] = df['balance'].apply(to_float)
    
    # Calculate expected balance
    if opening_balance is None and len(df) > 0:
        # Try to infer opening balance from first row
        first_balance = df.iloc[0]['balance_amount']
        first_debit = df.iloc[0]['debit_amount']
        first_credit = df.iloc[0]['credit_amount']
        opening_balance = first_balance + first_debit - first_credit
    
    running_balance = opening_balance if opening_balance else 0
    df['calculated_balance'] = 0.0
    df['balance_match'] = True
    
    for idx, row in df.iterrows():
        running_balance = running_balance - row['debit_amount'] + row['credit_amount']
        df.at[idx, 'calculated_balance'] = running_balance
        
        # Check if balances match (within tolerance for rounding)
        if row['balance_amount'] > 0:
            diff = abs(running_balance - row['balance_amount'])
            df.at[idx, 'balance_match'] = diff < 0.01
    
    return df

In [10]:
# Load ground truth data for evaluation
def load_ground_truth():
    """
    Load ground truth CSV and filter for bank statements.
    """
    import csv
    
    ground_truth = {}
    
    with open(GROUND_TRUTH_CSV, 'r') as f:
        reader = csv.DictReader(f)
        
        for row in reader:
            # Only process STATEMENT document types
            if row['DOCUMENT_TYPE'] == 'STATEMENT':
                image_file = row['image_file']
                
                # Extract key fields for bank statements
                ground_truth[image_file] = {
                    'bank_name': row['SUPPLIER_NAME'],  # Bank name mapped to supplier
                    'account_holder': row['PAYER_NAME'],  # Account holder mapped to payer
                    'transaction_dates': row['TRANSACTION_DATES'].split(' | ') if row['TRANSACTION_DATES'] != 'NOT_FOUND' else [],
                    'transaction_amounts_paid': row['TRANSACTION_AMOUNTS_PAID'].split(' | ') if row['TRANSACTION_AMOUNTS_PAID'] != 'NOT_FOUND' else [],
                    'document_type': row['DOCUMENT_TYPE']
                }
    
    return ground_truth

# Load ground truth
print("📊 Loading ground truth data...")
ground_truth_data = load_ground_truth()

print(f"Found {len(ground_truth_data)} bank statements in ground truth:")
for img_file, data in ground_truth_data.items():
    if 'statement' in img_file:
        print(f"  • {img_file}: {data['bank_name']} - {data['account_holder']}")
        print(f"    Transactions: {len(data['transaction_dates'])} dates, {len([amt for amt in data['transaction_amounts_paid'] if amt != 'NOT_FOUND'])} debits")

📊 Loading ground truth data...
Found 5 bank statements in ground truth:
  • commbank_statement_001.png: Commonwealth Bank of Australia - DAVID ROBERT BROWN
    Transactions: 19 dates, 16 debits
  • anz_statement_001.png: Australia and New Zealand Banking Group - CHRISTOPHER PAUL WHITE
    Transactions: 17 dates, 15 debits
  • nab_statement_001.png: National Australia Bank - CHRISTOPHER PAUL WHITE
    Transactions: 28 dates, 21 debits
  • westpac_statement_001.png: Westpac Banking Corporation - MARK STEVEN ANDERSON
    Transactions: 17 dates, 14 debits


## Extract Bank Statement Transactions

In [11]:
# Try the enhanced extraction for complex statements with multiple sections
print("📄 Extracting COMPLETE bank statement with all sections...")
print("=" * 60)

# Use the enhanced prompt for complex statements
complex_result = extract_bank_statement(
    STATEMENT_IMAGE_PATH,
    prompt_type="full_statement_extraction",
    custom_prompt=COMPLEX_STATEMENT_PROMPTS["full_statement_extraction"]
)

print("\n📊 Full Statement Extraction Response (first 2000 chars):")
print("-" * 60)
print(complex_result["raw_response"][:2000])
print("..." if len(complex_result["raw_response"]) > 2000 else "")
print("-" * 60)

📄 Extracting COMPLETE bank statement with all sections...
🔍 Analyzing bank statement...

📊 Full Statement Extraction Response (first 2000 chars):
------------------------------------------------------------
The image presents a comprehensive bank statement for the Commonwealth Bank of Australia, detailing transactions from August 7, 2025, to September 6, 2025. The statement is organized into three main sections: the header/account summary, the transaction table, and the summary/totals.

**HEADER/ACCOUNT SUMMARY**

* **Bank Name and Logo Area**: Commonwealth Bank of Australia
* **Account Holder Name**: David Robert Brown
* **Account Number**: 979918123 (partially masked)
* **Statement Period**: August 7, 2025, to September 6, 2025
* **Opening Balance**: $9188.50 CR
* **Total Deposits/Credits for Period**: $8586.28 CR
* **Total Withdrawals/Debits for Period**: $9188.50 CR
* **Closing Balance**: $8586.28 CR

**TRANSACTION TABLE**

| Date | Transaction | Debit | Credit | Balance |
| --- | 

In [12]:
# Parse the complex statement response
complex_data = parse_statement_json(complex_result["raw_response"])

if complex_data:
    print("✅ Successfully parsed complex bank statement\n")
    
    # Display all sections
    if "header" in complex_data:
        print("🏦 HEADER/ACCOUNT INFORMATION:")
        for key, value in complex_data["header"].items():
            print(f"  {key.replace('_', ' ').title()}: {value}")
    
    if "account_details" in complex_data:
        print("\n📋 ACCOUNT SUMMARY:")
        for key, value in complex_data["account_details"].items():
            print(f"  {key.replace('_', ' ').title()}: {value}")
    
    if "transactions" in complex_data:
        trans_df = pd.DataFrame(complex_data["transactions"])
        print(f"\n📊 TRANSACTION TABLE: {len(trans_df)} transactions")
        print("\nFirst 10 transactions:")
        display(trans_df.head(10))
        
        # Check for empty debit/credit handling
        has_nulls = trans_df[['debit', 'credit']].isnull().any().any()
        print(f"\n✅ Empty cell handling: {'Correct (nulls present)' if has_nulls else 'Check - no nulls detected'}")
    
    if "summary" in complex_data:
        print("\n📈 SUMMARY SECTION:")
        for key, value in complex_data["summary"].items():
            print(f"  {key.replace('_', ' ').title()}: {value}")
else:
    print("⚠️ Parsing as JSON failed, trying structured text extraction...")

⚠️ Parsing as JSON failed, trying structured text extraction...


In [13]:
# Extract using structured JSON format
print("📄 Extracting bank statement transactions...")
print("=" * 60)

result = extract_bank_statement(
    STATEMENT_IMAGE_PATH, 
    prompt_type="structured_transactions"
)

print("\n📊 Raw Response (first 1000 chars):")
print("-" * 60)
print(result["raw_response"][:1000])
print("..." if len(result["raw_response"]) > 1000 else "")
print("-" * 60)

📄 Extracting bank statement transactions...
🔍 Analyzing bank statement...

📊 Raw Response (first 1000 chars):
------------------------------------------------------------
Sure, here is the data extracted from the bank statement in a structured table format as requested:

| Date | Description | Debit | Credit | Balance |
| --- | --- | --- | --- | --- |
| Thu 04 Sep 2025 | Direct Debit DOMINO'S PTY LTD | $117.57 |  | $8586.28 CR |
| Mon 01 Sep 2025 | Monthly Service Fee |  | $17.89 | $8703.85 CR |
| Sun 31 Aug 2025 | Monthly Service Fee |  | $18.87 | $8721.74 CR |
| Fri 29 Aug 2025 | Wdi ATM WBC WESTPAC GLEN WAVE | $241.14 |  | $8740.61 CR |
| Wed 27 Aug 2025 | Direct Debit 94924P40133289 MHF 75969 Cash Withdrawal ATM SYDNEY NSW | $251.33 |  | $9080.28 CR |
| Sat 23 Aug 2025 | Home Loan Payment LN REPAY 56052P41700670 |  | $1918.75 | $9331.81 CR |
| Thu 21 Aug 2025 | Salary - ATO PAYROLL 89278P11488578 Direct Debit COLES PTY LTD | $7836.75 |  | $11526.67 CR |
| Tue 19 Aug 2025 | Cash Wit

In [14]:
# Parse and display structured data
parsed_data = parse_statement_json(result["raw_response"])

if parsed_data:
    print("✅ Successfully parsed bank statement data\n")
    
    # Display account details
    if "account_details" in parsed_data:
        print("🏦 Account Details:")
        for key, value in parsed_data["account_details"].items():
            print(f"  {key.replace('_', ' ').title()}: {value}")
    
    # Convert transactions to DataFrame
    if "transactions" in parsed_data:
        transactions_df = pd.DataFrame(parsed_data["transactions"])
        
        print(f"\n📊 Transactions: {len(transactions_df)} found")
        print("\nFirst 5 transactions:")
        display(transactions_df.head())
        
        # Save to CSV
        output_file = "extracted_bank_transactions.csv"
        transactions_df.to_csv(output_file, index=False)
        print(f"\n💾 Saved to {output_file}")
else:
    print("⚠️ Could not parse JSON. Trying row-by-row format...")

✅ Successfully parsed bank statement data

🏦 Account Details:
  Account Number: 979918123
  Statement Period: 07 Aug 2025 to 06 Sep 2025
  Opening Balance: $9188.50 CR
  Closing Balance: $8586.28 CR

📊 Transactions: 13 found

First 5 transactions:


,date,description,debit,credit,balance
0,Thu 04 Sep 2025,Direct Debit DOMINO'S PTY LTD,$117.57,None,$8586.28 CR
1,Mon 01 Sep 2025,Monthly Service Fee,None,$17.89,$8703.85 CR
2,Sun 31 Aug 2025,Monthly Service Fee,None,$18.87,$8721.74 CR
3,Fri 29 Aug 2025,Wdi ATM WBC WESTPAC GLEN WAVE,$241.14,None,$8740.61 CR
4,Wed 27 Aug 2025,Direct Debit 94924P40133289 MHF 75969 Cash Wit...,$251.33,None,$9080.28 CR



💾 Saved to extracted_bank_transactions.csv


In [15]:
# Try row-by-row extraction for better structure preservation
print("\n🔍 Trying row-by-row extraction...")
row_result = extract_bank_statement(
    STATEMENT_IMAGE_PATH,
    prompt_type="row_by_row_extraction"
)

print("\n📊 Row Format Response (first 1500 chars):")
print("-" * 60)
print(row_result["raw_response"][:1500])
print("-" * 60)

# Parse row format
row_df = parse_row_format(row_result["raw_response"])
if row_df is not None:
    print(f"\n✅ Extracted {len(row_df)} transaction rows")
    print("\nTransaction Table:")
    display(row_df)
    
    # Use this DataFrame for further processing
    transactions_df = row_df


🔍 Trying row-by-row extraction...
🔍 Analyzing bank statement...

📊 Row Format Response (first 1500 chars):
------------------------------------------------------------
**COLUMNS:**

| Date | Description | Debit | Credit | Balance |
| --- | --- | --- | --- | --- |
| Thu 04 Sep 2025 | Direct Debit DOMINO'S PTY LTD | $117.57 |  | $8586.28 CR |
| Mon 01 Sep 2025 | Monthly Service Fee |  | $17.89 | $8703.85 CR |
| Sun 31 Aug 2025 | Monthly Service Fee |  | $18.87 | $8721.74 CR |
| Fri 29 Aug 2025 | Wdi ATM WBC WESTPAC GLEN WAVE | $241.14 |  | $8740.61 CR |
| Wed 27 Aug 2025 | Direct Debit 94924P40133289 MHF 75969 Cash Withdrawal ATM SYDNEY NSW | $251.33 | $98.53 | $9080.28 CR |
| Sat 23 Aug 2025 | Home Loan Payment LN REPAY 56052P41700670 | $1918.75 |  | $9331.81 CR |
| Thu 21 Aug 2025 | Salary - ATO PAYROLL 89278P11488578 Direct Debit COLES PTY LTD | $7836.75 |  | $11526.67 CR |
| Tue 19 Aug 2025 | Cash Withdrawal ATM HOBART TAS | $61.68 |  | $3689.92 CR |
| Sun 17 Aug 2025 | Transfer To 

In [16]:
# Validate transactions
if 'transactions_df' in locals() and transactions_df is not None:
    print("\n🔍 Validating Transactions...")
    print("=" * 60)
    
    validation_results = validate_transactions(transactions_df)
    
    if validation_results["valid"]:
        print("✅ All transactions passed validation!")
    else:
        print("⚠️ Validation issues found:")
        for issue in validation_results["issues"]:
            print(f"  - {issue}")
    
    print(f"\n📊 Transaction Summary:")
    print(f"  Total Transactions: {validation_results['transaction_count']}")
    print(f"  Total Debits: ${validation_results['total_debits']:,.2f}")
    print(f"  Total Credits: ${validation_results['total_credits']:,.2f}")
    print(f"  Net Change: ${validation_results['net_change']:,.2f}")


🔍 Validating Transactions...
✅ All transactions passed validation!

📊 Transaction Summary:
  Total Transactions: 13
  Total Debits: $0.00
  Total Credits: $0.00
  Net Change: $0.00


In [17]:
# Calculate and verify running balance
if 'transactions_df' in locals() and transactions_df is not None:
    print("\n💰 Calculating Running Balance...")
    print("=" * 60)
    
    # Calculate with balance verification
    balanced_df = calculate_running_balance(transactions_df)
    
    # Show results
    print("\nTransactions with Balance Verification:")
    display_cols = ['date', 'description', 'debit', 'credit', 'balance', 'calculated_balance', 'balance_match']
    display_cols = [col for col in display_cols if col in balanced_df.columns]
    display(balanced_df[display_cols].head(10))
    
    # Check for mismatches
    mismatches = balanced_df[balanced_df['balance_match'] == False]
    if len(mismatches) > 0:
        print(f"\n⚠️ Found {len(mismatches)} balance mismatches:")
        display(mismatches[display_cols])
    else:
        print("\n✅ All balances match calculated values!")
    
    # Save enhanced version
    balanced_df.to_csv("bank_transactions_with_validation.csv", index=False)
    print("\n💾 Saved validated transactions to bank_transactions_with_validation.csv")


💰 Calculating Running Balance...

Transactions with Balance Verification:


,date,description,debit,credit,balance,calculated_balance,balance_match
0,Thu 04 Sep 2025,Direct Debit DOMINO'S PTY LTD,$117.57,None,$8586.28 CR,0.00,True
1,Mon 01 Sep 2025,Monthly Service Fee,None,$17.89,$8703.85 CR,17.89,True
2,Sun 31 Aug 2025,Monthly Service Fee,None,$18.87,$8721.74 CR,36.76,True
3,Fri 29 Aug 2025,Wdi ATM WBC WESTPAC GLEN WAVE,$241.14,None,$8740.61 CR,-204.38,True
4,Wed 27 Aug 2025,Direct Debit 94924P40133289 MHF 75969 Cash Wit...,$251.33,None,$9080.28 CR,-455.71,True
5,Sat 23 Aug 2025,Home Loan Payment LN REPAY 56052P41700670,None,$1918.75,$9331.81 CR,1463.04,True
6,Thu 21 Aug 2025,Salary - ATO PAYROLL 89278P11488578 Direct Deb...,$7836.75,None,$11526.67 CR,-6373.71,True
7,Tue 19 Aug 2025,Cash Withdrawal ATM HOBART TAS,$61.68,None,$3689.92 CR,-6435.39,True
8,Sun 17 Aug 2025,Transfer To Vicks Account NetBank From Tod,None,$3775.28,$3751.80 CR,-2660.11,True
9,Sat 16 Aug 2025,Salary Payment ATO 65208P39971825,None,$5778.51,$7526.88 CR,3118.40,True



✅ All balances match calculated values!

💾 Saved validated transactions to bank_transactions_with_validation.csv


In [18]:
# Advanced: Extract with custom prompt for specific bank formats
custom_bank_prompt = """
Extract the transaction table from this bank statement.

CRITICAL: This is a TABULAR structure where:
- Each row represents exactly ONE transaction
- Debit column shows withdrawals/payments (empty for deposits)
- Credit column shows deposits/receipts (empty for withdrawals)
- One of debit/credit MUST be empty for each transaction

For each transaction row, extract these columns IN ORDER:
1. Date (as shown)
2. Transaction details/description
3. Reference number (if present, else "N/A")
4. Debit amount (if present, else "0.00")
5. Credit amount (if present, else "0.00")  
6. Balance

Format as pipe-separated values:
Date|Description|Reference|Debit|Credit|Balance
[actual data rows follow]

Preserve EXACT row structure - do not combine or skip rows.
"""

print("\n🎯 Using custom extraction prompt...")
custom_result = extract_bank_statement(
    STATEMENT_IMAGE_PATH,
    custom_prompt=custom_bank_prompt
)

print("\nCustom extraction result (first 800 chars):")
print(custom_result["raw_response"][:800])


🎯 Using custom extraction prompt...
🔍 Analyzing bank statement...

Custom extraction result (first 800 chars):
**Transaction Table Extraction**

Here is the extracted transaction table from the bank statement:

| Date | Transaction Details/Description | Reference | Debit | Credit | Balance |
| --- | --- | --- | --- | --- | --- |
| Thu 04 Sep 2025 | Direct Debit DOMINO'S PTY LTD | $117.57 |  |  | $8586.28 CR |
| Mon 01 Sep 2025 | Monthly Service Fee |  | $17.89 |  | $8703.85 CR |
| Sun 31 Aug 2025 | Monthly Service Fee |  |  | $18.87 | $8721.74 CR |
| Fri 29 Aug 2025 | Wdi ATM WBC WESTPAC GLEN WAVE |  | $241.14 |  | $8740.61 CR |
| Wed 27 Aug 2025 | Direct Debit 94924P40133289 MHF 75969 |  | $251.33 |  | $9080.28 CR |
| Sat 23 Aug 2025 | Cash Withdrawal ATM SYDNEY NSW |  | $98.53 |  | $8981.75 CR |
| Thu 21 Aug 2025 | Home Loan Payment LN REPAY 56052P41700670 |  | $1918.75 |  | $9331.81 CR |
| Tue 1


In [19]:
# Generate evaluation summary report
def generate_evaluation_report(results, evaluation):
    """
    Generate a comprehensive evaluation report.
    """
    print("\n" + "="*70)
    print("📊 LLAMA VISION BANK STATEMENT EXTRACTION - EVALUATION REPORT")
    print("="*70)
    
    # Overall statistics
    total_banks = len(results)
    successful_extractions = sum(1 for r in results.values() if r['success'])
    
    print(f"\n📈 OVERALL STATISTICS:")
    print(f"  Total Banks Processed: {total_banks}")
    print(f"  Successful Extractions: {successful_extractions}/{total_banks}")
    print(f"  Success Rate: {(successful_extractions/total_banks)*100:.1f}%")
    
    # Field-level accuracy
    if evaluation:
        bank_name_matches = sum(1 for e in evaluation.values() if e.get('bank_name_match', False))
        holder_matches = sum(1 for e in evaluation.values() if e.get('account_holder_match', False))
        count_matches = sum(1 for e in evaluation.values() if e.get('transaction_count_match', False))
        avg_date_match = sum(e.get('dates_match_ratio', 0) for e in evaluation.values()) / len(evaluation) * 100
        
        print(f"\n🎯 FIELD-LEVEL ACCURACY:")
        print(f"  Bank Name Recognition: {bank_name_matches}/{total_banks} ({(bank_name_matches/total_banks)*100:.1f}%)")
        print(f"  Account Holder Extraction: {holder_matches}/{total_banks} ({(holder_matches/total_banks)*100:.1f}%)")
        print(f"  Transaction Count Accuracy: {count_matches}/{total_banks} ({(count_matches/total_banks)*100:.1f}%)")
        print(f"  Average Date Match Rate: {avg_date_match:.1f}%")
    
    # Detailed results per bank
    print(f"\n📋 DETAILED RESULTS BY BANK:")
    print("-"*70)
    
    for bank_name in BANK_STATEMENT_IMAGES.keys():
        if bank_name not in results:
            continue
            
        result = results[bank_name]
        eval_data = evaluation.get(bank_name, {})
        
        print(f"\n🏦 {bank_name}")
        print(f"  Status: {'✅ Success' if result['success'] else '❌ Failed'}")
        
        if result['success']:
            print(f"  Transactions Extracted: {result.get('transaction_count', 0)}")
            print(f"  Expected Transactions: {eval_data.get('expected_count', 'N/A')}")
            
            if eval_data:
                print(f"  Accuracy Metrics:")
                print(f"    - Bank Name: {'✅' if eval_data.get('bank_name_match') else '❌'}")
                print(f"    - Account Holder: {'✅' if eval_data.get('account_holder_match') else '❌'}")
                print(f"    - Transaction Count: {'✅' if eval_data.get('transaction_count_match') else '❌'}")
                print(f"    - Date Accuracy: {eval_data.get('dates_match_ratio', 0)*100:.1f}%")
        else:
            print(f"  Error: {result.get('error', 'Unknown error')}")
    
    print("\n" + "="*70)
    
    # Export results to CSV
    export_results_to_csv(results, evaluation)
    
    return {
        'total_banks': total_banks,
        'successful_extractions': successful_extractions,
        'field_accuracy': {
            'bank_name': bank_name_matches/total_banks if evaluation else 0,
            'account_holder': holder_matches/total_banks if evaluation else 0,
            'transaction_count': count_matches/total_banks if evaluation else 0,
            'date_match': avg_date_match/100 if evaluation else 0
        }
    }

def export_results_to_csv(results, evaluation):
    """
    Export extraction results to CSV matching evaluation format.
    """
    output_file = "llama_bank_extraction_results.csv"
    
    with open(output_file, 'w', newline='') as f:
        import csv
        
        fieldnames = ['Bank', 'Image', 'Success', 'Transactions_Extracted', 'Expected_Transactions',
                     'Bank_Name_Match', 'Account_Holder_Match', 'Count_Match', 'Date_Match_Rate']
        
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        
        for bank_name, result in results.items():
            eval_data = evaluation.get(bank_name, {})
            
            row = {
                'Bank': bank_name,
                'Image': result['image_path'].split('/')[-1],
                'Success': 'Yes' if result['success'] else 'No',
                'Transactions_Extracted': result.get('transaction_count', 0),
                'Expected_Transactions': eval_data.get('expected_count', 'N/A'),
                'Bank_Name_Match': 'Yes' if eval_data.get('bank_name_match') else 'No',
                'Account_Holder_Match': 'Yes' if eval_data.get('account_holder_match') else 'No',
                'Count_Match': 'Yes' if eval_data.get('transaction_count_match') else 'No',
                'Date_Match_Rate': f"{eval_data.get('dates_match_ratio', 0)*100:.1f}%" if eval_data else 'N/A'
            }
            
            writer.writerow(row)
    
    print(f"\n💾 Results exported to {output_file}")

# Generate the evaluation report
if 'all_results' in locals() and 'evaluation' in locals():
    report_summary = generate_evaluation_report(all_results, evaluation)
    
    # Overall performance score
    overall_score = (
        report_summary['field_accuracy']['bank_name'] * 0.2 +
        report_summary['field_accuracy']['account_holder'] * 0.2 +
        report_summary['field_accuracy']['transaction_count'] * 0.3 +
        report_summary['field_accuracy']['date_match'] * 0.3
    )
    
    print(f"\n🏆 OVERALL PERFORMANCE SCORE: {overall_score*100:.1f}%")
    
    if overall_score >= 0.8:
        print("   ✅ Excellent extraction performance!")
    elif overall_score >= 0.6:
        print("   ⚠️ Good performance, but room for improvement")
    else:
        print("   ❌ Significant accuracy issues - review prompts and model configuration")

In [20]:
# Batch processing for all Big 4 bank statements
def process_all_bank_statements(evaluate=True):
    """
    Process all Big 4 bank statements and optionally evaluate against ground truth.
    """
    results = {}
    
    for bank_name, image_path in BANK_STATEMENT_IMAGES.items():
        print(f"\n{'='*60}")
        print(f"🏦 Processing {bank_name} Statement")
        print(f"📄 Image: {image_path}")
        print('='*60)
        
        try:
            # Extract using structured format
            result = extract_bank_statement(
                image_path,
                prompt_type="structured_transactions"
            )
            
            # Parse response
            parsed_data = parse_statement_json(result["raw_response"])
            
            if parsed_data:
                # Extract key information
                extraction_result = {
                    'bank': bank_name,
                    'image_path': image_path,
                    'raw_response': result["raw_response"],
                    'parsed_data': parsed_data,
                    'success': True
                }
                
                # Extract specific fields if available
                if "account_details" in parsed_data:
                    extraction_result['account_holder'] = parsed_data["account_details"].get("account_holder", "NOT_EXTRACTED")
                    extraction_result['opening_balance'] = parsed_data["account_details"].get("opening_balance", "NOT_EXTRACTED")
                    extraction_result['closing_balance'] = parsed_data["account_details"].get("closing_balance", "NOT_EXTRACTED")
                
                if "transactions" in parsed_data:
                    transactions = parsed_data["transactions"]
                    extraction_result['transaction_count'] = len(transactions)
                    extraction_result['transactions'] = transactions
                    
                    # Extract dates and amounts
                    extraction_result['transaction_dates'] = [t.get('date', '') for t in transactions]
                    extraction_result['debit_amounts'] = [t.get('debit') for t in transactions if t.get('debit')]
                
                print(f"✅ Successfully extracted {extraction_result.get('transaction_count', 0)} transactions")
            else:
                extraction_result = {
                    'bank': bank_name,
                    'image_path': image_path,
                    'raw_response': result["raw_response"],
                    'success': False,
                    'error': 'Failed to parse JSON response'
                }
                print("❌ Failed to parse response")
            
            results[bank_name] = extraction_result
            
        except Exception as e:
            print(f"❌ Error processing {bank_name}: {e}")
            results[bank_name] = {
                'bank': bank_name,
                'image_path': image_path,
                'success': False,
                'error': str(e)
            }
    
    # Evaluate if requested
    if evaluate and ground_truth_data:
        print(f"\n{'='*60}")
        print("📊 EVALUATION RESULTS")
        print('='*60)
        evaluation_results = evaluate_extractions(results, ground_truth_data)
        return results, evaluation_results
    
    return results

def evaluate_extractions(extraction_results, ground_truth):
    """
    Compare extracted data with ground truth.
    """
    evaluation = {}
    
    for bank_name, extracted in extraction_results.items():
        if not extracted['success']:
            evaluation[bank_name] = {'success': False, 'error': extracted.get('error')}
            continue
        
        # Find corresponding ground truth entry
        image_file = extracted['image_path'].split('/')[-1]  # Get filename only
        
        if image_file not in ground_truth:
            evaluation[bank_name] = {'success': False, 'error': 'No ground truth found'}
            continue
        
        gt = ground_truth[image_file]
        
        # Compare key fields
        eval_result = {
            'bank_name_match': False,
            'account_holder_match': False,
            'transaction_count_match': False,
            'dates_match_ratio': 0.0
        }
        
        # Check bank name (might be in raw response if not parsed)
        if bank_name.lower() in extracted.get('raw_response', '').lower():
            eval_result['bank_name_match'] = True
        
        # Check account holder
        extracted_holder = extracted.get('account_holder', '').upper()
        gt_holder = gt['account_holder'].upper() if gt['account_holder'] != 'NOT_FOUND' else ''
        if extracted_holder and gt_holder and extracted_holder in gt_holder or gt_holder in extracted_holder:
            eval_result['account_holder_match'] = True
        
        # Check transaction counts
        extracted_count = extracted.get('transaction_count', 0)
        gt_count = len(gt['transaction_dates'])
        eval_result['transaction_count_match'] = (extracted_count == gt_count)
        eval_result['extracted_count'] = extracted_count
        eval_result['expected_count'] = gt_count
        
        # Check date overlap
        if extracted.get('transaction_dates'):
            extracted_dates = set(extracted['transaction_dates'])
            gt_dates = set(gt['transaction_dates'])
            if gt_dates:
                overlap = len(extracted_dates & gt_dates)
                eval_result['dates_match_ratio'] = overlap / len(gt_dates)
        
        evaluation[bank_name] = eval_result
        
        # Print results
        print(f"\n{bank_name}:")
        print(f"  Bank Name Match: {'✅' if eval_result['bank_name_match'] else '❌'}")
        print(f"  Account Holder Match: {'✅' if eval_result['account_holder_match'] else '❌'}")
        print(f"  Transaction Count: {extracted_count}/{gt_count} {'✅' if eval_result['transaction_count_match'] else '❌'}")
        print(f"  Date Match Rate: {eval_result['dates_match_ratio']*100:.1f}%")
    
    return evaluation

# Process all bank statements
print("🚀 Processing all Big 4 Australian Bank Statements...")
all_results, evaluation = process_all_bank_statements(evaluate=True)

🚀 Processing all Big 4 Australian Bank Statements...

🏦 Processing CommBank Statement
📄 Image: evaluation_data/commbank_statement_001.png
🔍 Analyzing bank statement...
✅ Successfully extracted 13 transactions

🏦 Processing ANZ Statement
📄 Image: evaluation_data/anz_statement_001.png
🔍 Analyzing bank statement...
✅ Successfully extracted 11 transactions

🏦 Processing NAB Statement
📄 Image: evaluation_data/nab_statement_001.png
🔍 Analyzing bank statement...
✅ Successfully extracted 14 transactions

🏦 Processing Westpac Statement
📄 Image: evaluation_data/westpac_statement_001.png
🔍 Analyzing bank statement...
✅ Successfully extracted 12 transactions

📊 EVALUATION RESULTS

CommBank:
  Bank Name Match: ❌
  Account Holder Match: ❌
  Transaction Count: 13/19 ❌
  Date Match Rate: 0.0%

ANZ:
  Bank Name Match: ❌
  Account Holder Match: ❌
  Transaction Count: 11/17 ❌
  Date Match Rate: 0.0%

NAB:
  Bank Name Match: ❌
  Account Holder Match: ❌
  Transaction Count: 14/28 ❌
  Date Match Rate: 0.0%


In [21]:
# Batch processing for multiple statement pages
def process_multi_page_statement(image_paths: List[str]) -> pd.DataFrame:
    """
    Process multiple pages of a bank statement.
    """
    all_transactions = []
    
    for i, image_path in enumerate(image_paths, 1):
        print(f"\n📄 Processing page {i}/{len(image_paths)}...")
        
        try:
            # Extract transactions
            result = extract_bank_statement(image_path, "structured_transactions")
            parsed = parse_statement_json(result["raw_response"])
            
            if parsed and "transactions" in parsed:
                page_df = pd.DataFrame(parsed["transactions"])
                page_df['page'] = i
                all_transactions.append(page_df)
                print(f"  ✅ Extracted {len(page_df)} transactions")
            else:
                print(f"  ⚠️ No transactions found")
                
        except Exception as e:
            print(f"  ❌ Error: {e}")
    
    if all_transactions:
        combined_df = pd.concat(all_transactions, ignore_index=True)
        print(f"\n✅ Total transactions extracted: {len(combined_df)}")
        return combined_df
    
    return pd.DataFrame()

# Example usage (uncomment and update paths):
# statement_pages = ["statement_page1.jpg", "statement_page2.jpg"]
# multi_page_df = process_multi_page_statement(statement_pages)
# multi_page_df.to_csv("complete_statement.csv", index=False)

In [23]:
# Export to different formats
if 'transactions_df' in locals() and transactions_df is not None:
    print("\n📤 Exporting Transactions...")
    
    # 1. Excel format with formatting
    with pd.ExcelWriter('bank_transactions.xlsx', engine='openpyxl') as writer:
        transactions_df.to_excel(writer, sheet_name='Transactions', index=False)
        
        # Add summary sheet if validation was done
        if 'validation_results' in locals():
            summary_df = pd.DataFrame([{
                'Total Transactions': validation_results['transaction_count'],
                'Total Debits': f"${validation_results['total_debits']:,.2f}",
                'Total Credits': f"${validation_results['total_credits']:,.2f}",
                'Net Change': f"${validation_results['net_change']:,.2f}"
            }])
            summary_df.T.to_excel(writer, sheet_name='Summary')
    
    print("  ✅ Saved to bank_transactions.xlsx")
    
    # 2. JSON format for APIs
    transactions_json = transactions_df.to_json(orient='records', indent=2)
    with open('bank_transactions.json', 'w') as f:
        f.write(transactions_json)
    print("  ✅ Saved to bank_transactions.json")
    
    # 3. Accounting software format (QIF)
    qif_output = "!Type:Bank\n"
    for _, row in transactions_df.iterrows():
        qif_output += f"D{row.get('date', '')}\n"
        qif_output += f"P{row.get('description', '')}\n"
        
        # Amount (negative for debits)
        if row.get('debit') and str(row['debit']).replace('.','').isdigit():
            qif_output += f"T-{row['debit']}\n"
        elif row.get('credit') and str(row['credit']).replace('.','').isdigit():
            qif_output += f"T{row['credit']}\n"
        qif_output += "^\n"
    
    with open('bank_transactions.qif', 'w') as f:
        f.write(qif_output)
    print("  ✅ Saved to bank_transactions.qif (Quicken format)")
    
    print("\n📊 Export complete!")


📤 Exporting Transactions...
  ✅ Saved to bank_transactions.xlsx
  ✅ Saved to bank_transactions.json
  ✅ Saved to bank_transactions.qif (Quicken format)

📊 Export complete!


In [24]:
# Clean up GPU memory
def cleanup_gpu():
    """
    Clean up GPU memory after processing.
    """
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("🧹 GPU memory cleaned")
        print(f"   Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        print(f"   Reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB")

# Uncomment to clean up
# cleanup_gpu()

## Tips for Bank Statement Extraction

### Key Challenges & Solutions

1. **Empty Debit/Credit Cells**
   - The prompts explicitly instruct to mark empty cells
   - Each transaction has EITHER debit OR credit, never both
   - Validation checks ensure this rule is followed

2. **Maintaining Row Structure**
   - Prompts emphasize treating statements as tables
   - Row-by-row extraction preserves exact structure
   - No skipping or combining of rows

3. **Balance Verification**
   - Automatic calculation of expected balances
   - Comparison with extracted balances
   - Highlights any discrepancies

### Best Practices

- **Image Quality**: Higher resolution improves accuracy
- **Clear Tables**: Well-defined borders help extraction
- **Multiple Formats**: Try different prompt types for best results
- **Validation**: Always validate debit/credit exclusivity
- **Multi-Page**: Process each page separately, then combine

### Common Bank Statement Formats

- **Standard**: Date | Description | Debit | Credit | Balance
- **With Reference**: Date | Description | Reference | Debit | Credit | Balance  
- **Combined Amount**: Date | Description | Amount | Balance (use +/- signs)
- **Separate Columns**: Date | Withdrawals | Deposits | Balance

### Export Options

- **CSV**: Universal format for spreadsheets
- **Excel**: With formatting and summary sheets
- **JSON**: For APIs and web applications
- **QIF**: For accounting software import